# 03 - Data Preparation (CRISP-DM)
Feature engineering, missing-value strategy, NLP text features and the **train / validation / test split**.

Regulatory design rules applied here:
- **Every imputation/encoding decision is logged** (`imputation_log`, `feature_catalog`) with a business rationale.
- **No leakage:** the split happens first; provider/attorney fraud rates and text vocabularies are learned on *train only* (out-of-fold for train rows).
- Geo-clustering uses only *prior* claims (what the claims system would know at scoring time) and no labels.
- Protected/demographic fields (`insured_age`, `state`) are **excluded from model features** (kept as context columns for the fairness audit). Flip `INCLUDE_DEMOGRAPHICS` to test.

Outputs: `features_all`, `feature_catalog`, `provider_rate_lookup`, `text_vocab`, `imputation_log`.

In [0]:
SRC_TABLE = "workspace.insurance_fraud.auto_insurance_fraud"   # READ-ONLY warehouse table
DM = "workspace.dm_final_project"                               # ALL derived work goes here
SEED = 42
assert "insurance_fraud" not in DM, "Never write to the warehouse schema"

import numpy as np, pandas as pd, re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import chi2
from sklearn.neighbors import BallTree

INCLUDE_DEMOGRAPHICS = False
ORG, OPP = "is_organized_fraud", "is_opportunistic_fraud"
GEO_RADIUS_KM, GEO_WINDOW_DAYS = 1.0, 30
SMOOTH_M = 20          # smoothing strength for target-encoded provider/attorney rates
N_TEXT_TERMS = 25      # data-driven n-grams kept per target

df = spark.table(f"{DM}.claims_clean").toPandas()
df["claim_date"] = pd.to_datetime(df["claim_date"])
df = df.sort_values(["claim_date", "claim_id"]).reset_index(drop=True)
print(df.shape)

## 1. Train / validation / test split (60/20/20, stratified on both targets)

In [0]:
strata = df[ORG] * 2 + df[OPP]
strata = strata.where(strata.map(strata.value_counts()) >= 10, 0)   # fold ultra-rare combos
idx_train, idx_tmp = train_test_split(df.index, test_size=0.4, stratify=strata, random_state=SEED)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, stratify=strata.loc[idx_tmp], random_state=SEED)
df["split"] = "train"; df.loc[idx_val, "split"] = "val"; df.loc[idx_test, "split"] = "test"
display(df.groupby("split")[[ORG, OPP]].agg(["count", "mean"]).round(4))

## 2. Row-level behavioral features

In [0]:
BASE_DESC = {}   # feature -> (group, description, rationale)
def reg(name, group, desc):
    BASE_DESC[name] = (group, desc)

def row_features(d):
    f = pd.DataFrame(index=d.index)
    f["claim_id"] = d["claim_id"]
    # --- Temporal
    f["report_delay_hours"] = d["report_delay_hours"];                         reg("report_delay_hours", "temporal", "Hours between incident and report")
    f["log_report_delay"] = np.log1p(d["report_delay_hours"].clip(lower=0));   reg("log_report_delay", "temporal", "log1p of report delay (tames skew)")
    f["policy_inception_days"] = d["policy_inception_days"];                   reg("policy_inception_days", "temporal", "Days since policy purchase")
    f["new_policy_30d"] = (d["policy_inception_days"] <= 30).astype(int);      reg("new_policy_30d", "temporal", "Policy bought <=30 days before claim")
    f["incident_time_hour"] = d["incident_time_hour"];                         reg("incident_time_hour", "temporal", "Hour of incident (0-23)")
    f["night_incident"] = d["incident_time_hour"].isin([23, 0, 1, 2, 3, 4, 5]).astype(int); reg("night_incident", "temporal", "Incident between 23:00 and 05:59")
    f["claim_month"] = d["claim_date"].dt.month;                               reg("claim_month", "temporal", "Calendar month of claim (seasonality)")
    f["claim_dow"] = d["claim_date"].dt.dayofweek;                             reg("claim_dow", "temporal", "Day of week of claim")
    # --- Financial
    init = d["initial_damage_amount"].where(d["initial_damage_amount"].notna(), d["damage_amount"])  # not amended -> no inflation
    f["initial_damage_missing"] = d["initial_damage_amount"].isna().astype(int);  reg("initial_damage_missing", "financial", "No initial estimate on file (claim never amended)")
    f["damage_inflation_abs"] = d["damage_amount"] - init;                     reg("damage_inflation_abs", "financial", "Final minus initial damage estimate (USD)")
    ratio = d["damage_amount"] / init.replace(0, np.nan)
    f["damage_inflation_ratio"] = ratio.fillna(1.0);                           reg("damage_inflation_ratio", "financial", "Final / initial damage estimate")
    f["damage_inflated_25pct"] = (f["damage_inflation_ratio"] >= 1.25).astype(int); reg("damage_inflated_25pct", "financial", "Estimate grew >=25%")
    f["claim_amendments"] = d["claim_amendments"];                             reg("claim_amendments", "financial", "Times the claim story/amount was modified")
    f["log_damage_amount"] = np.log1p(d["damage_amount"].clip(lower=0));       reg("log_damage_amount", "financial", "log1p of final damage amount")
    f["settlement_inquiries"] = d["settlement_inquiries"];                     reg("settlement_inquiries", "financial", "Settlement-amount inquiries (financial motivation)")
    # --- Medical / legal
    f["medical_visits_count"] = d["medical_visits_count"];                     reg("medical_visits_count", "medical", "Number of medical appointments")
    f["has_medical_provider"] = d["medical_provider"].notna().astype(int);     reg("has_medical_provider", "medical", "A medical provider is on the claim")
    f["has_attorney"] = d["attorney_firm"].notna().astype(int);                reg("has_attorney", "legal", "Attorney retained")
    f["days_to_attorney"] = d["days_to_attorney"];                             reg("days_to_attorney", "legal", "Days from incident to attorney retention (imputed if none)")
    f["same_day_attorney"] = (d["days_to_attorney"] <= 1).astype(int);         reg("same_day_attorney", "legal", "Attorney retained within 1 day")
    # --- Occupancy
    f["num_people_in_car"] = d["num_people_in_car"];                           reg("num_people_in_car", "occupancy", "Occupants in vehicle")
    f["surname_ratio"] = d["num_different_surnames"] / d["num_people_in_car"].replace(0, np.nan); reg("surname_ratio", "occupancy", "Distinct surnames / occupants (unrelated-party signal)")
    f["unrelated_group"] = ((d["num_people_in_car"] >= 3) & (f["surname_ratio"] >= 0.75)).astype(int); reg("unrelated_group", "occupancy", ">=3 occupants and >=75% distinct surnames")
    # --- History / vehicle
    f["prior_claims_6mo"] = d["prior_claims_6mo"];                             reg("prior_claims_6mo", "history", "Claims in prior 6 months")
    f["vehicle_age_years"] = d["vehicle_age_years"];                           reg("vehicle_age_years", "history", "Vehicle age (years)")
    if INCLUDE_DEMOGRAPHICS:
        f["insured_age"] = d["insured_age"]; reg("insured_age", "demographic", "Age of insured (fairness-sensitive)")
    return f

base = row_features(df)
print(base.shape)

## 3. Geographic clustering (prior claims only, label-free)
For each claim: how many *earlier* claims (within the window) happened within the radius - and how many of those used the same provider / attorney. Staged-accident rings show up as dense, same-provider pockets.

In [0]:
def geo_features(d, radius_km=GEO_RADIUS_KM, window_days=GEO_WINDOW_DAYS):
    out = np.zeros((len(d), 4))
    valid = d[["incident_latitude", "incident_longitude"]].notna().all(axis=1).values
    out[:, 3] = (~valid).astype(int)
    sub = d.loc[valid].reset_index()          # 'index' = original position
    tree = BallTree(np.radians(sub[["incident_latitude", "incident_longitude"]].values), metric="haversine")
    neigh = tree.query_radius(np.radians(sub[["incident_latitude", "incident_longitude"]].values), r=radius_km / 6371.0)
    days = (sub["claim_date"].values.astype("datetime64[D]") - np.datetime64("1970-01-01", "D")).astype(int)
    prov = sub["medical_provider"].fillna("__NA__").values
    att = sub["attorney_firm"].fillna("__NA__").values
    for i, nb in enumerate(neigh):
        nb = nb[nb != i]
        if nb.size == 0: continue
        delta = days[i] - days[nb]
        nb = nb[(delta >= 0) & (delta <= window_days)]
        pos = sub.at[i, "index"]
        out[pos, 0] = nb.size
        out[pos, 1] = (prov[nb] == prov[i]).sum() if prov[i] != "__NA__" else 0
        out[pos, 2] = (att[nb] == att[i]).sum() if att[i] != "__NA__" else 0
    return pd.DataFrame(out, columns=["nearby_claims_30d", "nearby_same_provider_30d", "nearby_same_attorney_30d", "geo_missing"], index=d.index)

geo = geo_features(df)
reg("nearby_claims_30d", "network", f"Earlier claims within {GEO_RADIUS_KM} km in prior {GEO_WINDOW_DAYS} days")
reg("nearby_same_provider_30d", "network", "...of which used the same medical provider")
reg("nearby_same_attorney_30d", "network", "...of which used the same attorney")
reg("geo_missing", "network", "Incident coordinates missing (counts set to 0)")
base = pd.concat([base, geo], axis=1)
display(geo.describe().T)

## 4. Provider / attorney fraud rates (smoothed target encoding, leakage-safe)
Smoothed rate = (frauds + m x prior) / (claims + m). Train rows use **5-fold out-of-fold** rates so a claim never sees its own label; val/test use full-train rates. Unknown or missing provider/attorney -> global prior.

In [0]:
def smoothed_rate(keys_fit, y_fit, keys_apply, prior, m=SMOOTH_M):
    stats = pd.DataFrame({"k": keys_fit.values, "y": y_fit.values}).dropna(subset=["k"]).groupby("k")["y"].agg(["sum", "count"])
    s = keys_apply.map(stats["sum"]).fillna(0); n = keys_apply.map(stats["count"]).fillna(0)
    return (s + m * prior) / (n + m)

tr_mask = (df["split"] == "train").values
enc = pd.DataFrame(index=df.index)
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
tr_idx = np.where(tr_mask)[0]
ENC_SPECS = [("medical_provider", ORG, "provider_org_rate"), ("attorney_firm", ORG, "attorney_org_rate"),
             ("medical_provider", OPP, "provider_opp_rate"), ("attorney_firm", OPP, "attorney_opp_rate")]
for key, tgt, name in ENC_SPECS:
    prior = df.loc[tr_mask, tgt].mean()
    col = pd.Series(prior, index=df.index, dtype=float)
    # out-of-fold for train
    for fit_i, app_i in skf.split(tr_idx, df.loc[tr_mask, tgt]):
        fit_rows, app_rows = tr_idx[fit_i], tr_idx[app_i]
        col.iloc[app_rows] = smoothed_rate(df[key].iloc[fit_rows], df[tgt].iloc[fit_rows], df[key].iloc[app_rows], prior).values
    # full-train for val/test
    other = np.where(~tr_mask)[0]
    col.iloc[other] = smoothed_rate(df[key].iloc[tr_idx], df[tgt].iloc[tr_idx], df[key].iloc[other], prior).values
    enc[name] = col
    reg(name, "network", f"Smoothed historical {('organized' if tgt == ORG else 'opportunistic')} fraud rate of the claim's {key} (m={SMOOTH_M})")

for key, nm in [("medical_provider", "provider_log_volume"), ("attorney_firm", "attorney_log_volume")]:
    cnt = df.loc[tr_mask, key].value_counts()
    enc[nm] = np.log1p(df[key].map(cnt).fillna(0))
    reg(nm, "network", f"log1p(# train claims for the claim's {key}); 0 if unknown")
base = pd.concat([base, enc], axis=1)

# persist the production lookup (full-train statistics) for scoring-time joins
rows = []
for key, etype in [("medical_provider", "provider"), ("attorney_firm", "attorney")]:
    g = df.loc[tr_mask].groupby(key).agg(n_claims=(ORG, "count"), organized_frauds=(ORG, "sum"), opportunistic_frauds=(OPP, "sum")).reset_index().rename(columns={key: "entity"})
    g["entity_type"] = etype; rows.append(g)
lookup = pd.concat(rows)
lookup["org_rate_smoothed"] = (lookup["organized_frauds"] + SMOOTH_M * df.loc[tr_mask, ORG].mean()) / (lookup["n_claims"] + SMOOTH_M)
lookup["opp_rate_smoothed"] = (lookup["opportunistic_frauds"] + SMOOTH_M * df.loc[tr_mask, OPP].mean()) / (lookup["n_claims"] + SMOOTH_M)
spark.createDataFrame(lookup).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.provider_rate_lookup")
display(enc.describe().T)

## 5. NLP features from `claim_notes`
**(a) Domain-pattern flags** (regex over concepts from the business background) and **(b) data-driven n-grams** chosen by chi-square on *train only*, with a leakage filter that removes terms revealing the fraud outcome.

In [0]:
notes = df["claim_notes"].fillna("").str.lower()

PATTERNS = {
 "txt_all_parties_coordinated": r"\b(all|every|both)\b.{0,40}\b(passengers|occupants|parties|claimants)\b",
 "txt_same_facility":           r"same (medical )?(facility|clinic|provider|institute)",
 "txt_similar_injuries":        r"(similar|identical|same)\b.{0,30}\b(injur|symptom|complaint|pattern)",
 "txt_immediate_scene_treatment": r"(immediately|right away).{0,30}(medical|treatment|attention)|sought medical attention",
 "txt_template_severe":         r"severely impacted",
 "txt_witness_too_consistent":  r"(remarkably|extremely|unusually)\s+(detailed|consistent)|detailed and consistent",
 "txt_progressive_symptoms":    r"(initially|at first).{0,40}(fine|no pain|no injur)|developed (pain|symptoms)|developing (neck|back|pain)|worsen",
 "txt_financial_stress":        r"job loss|lost (his|her|their) job|financial (difficult|stress|hardship|trouble)|behind on (rent|bills)",
 "txt_settlement_focus":        r"settlement|typical (amount|payout)|how much",
 "txt_vague_symptoms":          r"hard to pinpoint|difficult to (describe|pinpoint)|vague|hard to describe",
 "txt_late_damage_discovery":   r"additional (vehicle )?damage|supplement|re-?inspect|discovered.{0,30}damage",
 "txt_attorney_mention":        r"attorney|lawyer|counsel",
 "txt_inconsistent_story":      r"inconsisten|changed (his|her|their) (story|account)|contradict",
}
txt = pd.DataFrame(index=df.index)
for name, pat in PATTERNS.items():
    txt[name] = notes.str.contains(pat, regex=True).astype(int)
    reg(name, "text_pattern", f"Regex flag in adjuster notes: {pat[:70]}")
txt["notes_word_count"] = notes.str.split().str.len().fillna(0); reg("notes_word_count", "text_pattern", "Words in adjuster notes")
txt["notes_missing"] = (notes == "").astype(int);               reg("notes_missing", "text_pattern", "No adjuster notes")
display(txt.drop(columns=["notes_word_count"]).groupby(df[ORG]).mean().T.rename(columns={0: "not_org", 1: "org"}).join(
        txt.drop(columns=["notes_word_count"]).groupby(df[OPP]).mean().T.rename(columns={0: "not_opp", 1: "opp"})).round(3))

In [0]:
LEAKY = re.compile(r"fraud|siu|euo|examination|denied|denial|referred|suspect|staged|ring\b|investigat|confirmed|scheme")
tr_notes = notes[tr_mask]
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=20, max_df=0.5, stop_words="english", sublinear_tf=True)
Xtr = tfidf.fit_transform(tr_notes)
vocab = np.array(tfidf.get_feature_names_out())
selected = {}
for tgt in [ORG, OPP]:
    scores, _ = chi2(Xtr, df.loc[tr_mask, tgt].values)
    scores = np.nan_to_num(scores)
    order = np.argsort(-scores)
    keep = [vocab[i] for i in order if not LEAKY.search(vocab[i])][:N_TEXT_TERMS]
    dropped = [vocab[i] for i in order[:N_TEXT_TERMS * 2] if LEAKY.search(vocab[i])]
    selected[tgt] = keep
    print(f"{tgt}: kept {len(keep)} terms; leakage-filtered: {dropped[:10]}")
    print("   ", keep)

terms = sorted(set(selected[ORG]) | set(selected[OPP]))
cv = CountVectorizer(ngram_range=(1, 2), vocabulary=terms, binary=True)
Xall = cv.transform(notes)
ngram = pd.DataFrame(Xall.toarray(), columns=["ng_" + re.sub(r"\W+", "_", t) for t in terms], index=df.index)
for c, t in zip(ngram.columns, terms):
    reg(c, "text_ngram", f"Adjuster notes contain n-gram '{t}' (chi-square selected on train)")
txt = pd.concat([txt, ngram], axis=1)
(spark.createDataFrame(pd.DataFrame({"term": terms, "feature": ngram.columns,
     "selected_for_org": [t in selected[ORG] for t in terms], "selected_for_opp": [t in selected[OPP] for t in terms]}))
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.text_vocab"))
base = pd.concat([base, txt], axis=1)

## 6. Missing-value strategy (train-derived, fully logged)

In [0]:
imp_log = []
FEATURES = [c for c in base.columns if c != "claim_id"]
Xtr_ = base.loc[tr_mask, FEATURES]

# Rule 1: days_to_attorney is missing because NO attorney was retained -> fill with the train 95th percentile ("very late") + has_attorney flag already exists
fill_days = Xtr_["days_to_attorney"].quantile(0.95)
n_na = int(base["days_to_attorney"].isna().sum())
base["days_to_attorney"] = base["days_to_attorney"].fillna(fill_days)
imp_log.append(dict(feature="days_to_attorney", n_missing=n_na, strategy=f"fill with train 95th pct = {fill_days:.1f}", rationale="Missing means no attorney yet; treated as 'very late' retention; has_attorney flag preserves the information"))

# Rule 2: every other feature with nulls -> train median + explicit missing-indicator
for c in FEATURES:
    n = int(base[c].isna().sum())
    if n == 0: continue
    med = Xtr_[c].median()
    base[f"{c}_missing"] = base[c].isna().astype(int)
    base[c] = base[c].fillna(med)
    reg(f"{c}_missing", "missing_indicator", f"{c} was missing before imputation")
    imp_log.append(dict(feature=c, n_missing=n, strategy=f"train median = {med:.3f} + {c}_missing flag", rationale="Value missing at random/unknown; median is robust to outliers; indicator keeps 'missingness' explainable"))
FEATURES = [c for c in base.columns if c != "claim_id"]
assert not base[FEATURES].isna().any().any(), "NaNs remain"
display(pd.DataFrame(imp_log) if imp_log else pd.DataFrame({"note": ["no further missing values"]}))

## 7. Assemble and persist

In [0]:
CONTEXT = ["claim_id", "split", ORG, OPP, "claim_date", "state", "insured_age", "damage_amount", "medical_provider", "attorney_firm"]
ctx = df[CONTEXT].copy(); ctx = ctx.rename(columns={"state": "ctx_state", "insured_age": "ctx_insured_age", "damage_amount": "ctx_damage_amount",
                                                     "medical_provider": "ctx_medical_provider", "attorney_firm": "ctx_attorney_firm"})
ctx["claim_date"] = ctx["claim_date"].dt.date
final = ctx.merge(base, on="claim_id", how="left")
final[FEATURES] = final[FEATURES].astype(float)
assert len(final) == len(df) and final["claim_id"].is_unique

sdf = spark.createDataFrame(final)
sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.features_all")

catalog = pd.DataFrame([dict(feature=f, group=BASE_DESC.get(f, ("other", ""))[0], description=BASE_DESC.get(f, ("", ""))[1]) for f in FEATURES])
spark.createDataFrame(catalog).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.feature_catalog")
if imp_log:
    spark.createDataFrame(pd.DataFrame(imp_log)).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.imputation_log")
print(f"{len(FEATURES)} features persisted")
display(catalog.groupby("group").size().rename("n_features").reset_index())
display(catalog)

In [0]:
# Sanity: single-feature signal on TRAIN only (AUC) - quick look for leakage / usefulness
from sklearn.metrics import roc_auc_score
tr = final[final["split"] == "train"]
rows = []
for f in FEATURES:
    if tr[f].nunique() < 2: continue
    rows.append(dict(feature=f, auc_org=roc_auc_score(tr[ORG], tr[f]), auc_opp=roc_auc_score(tr[OPP], tr[f])))
auc = pd.DataFrame(rows); auc["strength_org"] = (auc.auc_org - .5).abs(); auc["strength_opp"] = (auc.auc_opp - .5).abs()
display(auc.sort_values("strength_org", ascending=False).head(15).round(3))
display(auc.sort_values("strength_opp", ascending=False).head(15).round(3))
print("Any single feature with AUC > 0.95 is a LEAKAGE suspect:", auc[(auc.auc_org > .95) | (auc.auc_org < .05) | (auc.auc_opp > .95) | (auc.auc_opp < .05)].feature.tolist())